<a href="https://colab.research.google.com/github/Yusaku-KASAI/data-science-entrance/blob/colab/%E3%83%87%E3%83%BC%E3%82%BF%E3%82%B5%E3%82%A4%E3%82%A8%E3%83%B3%E3%82%B9%E8%B6%85%E5%85%A5%E9%96%80%E6%9C%80%E7%B5%82%E8%AA%B2%E9%A1%8C6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === （ノートブック用の任意前処理） =========================================
!rm -rf ~/.cache/matplotlib
!pip install -q pandas numpy requests statsmodels matplotlib scikit-learn
!apt-get -y update >/dev/null 2>&1 && apt-get -y install -qq fonts-noto-cjk >/dev/null 2>&1

In [ ]:
# -*- coding: utf-8 -*-
"""
賃金構造基本統計調査（統計表ID: 0003445758, 令和2年以降DB）
2020–2023を e-Stat API から取得 → 整形 → EDA → 回帰（県FE+年FE, クラスタロバスト）
→ 年別ITプレミアム推移 → 都道府県トレンド（%/年）
→ 都道府県クラスタ（KMeans, silhouetteでk自動選択）× IT相互作用（IT:C(cluster)）

・AppID: 環境変数 ESTAT_APP_ID（未設定時はデモ用キーを使用）
・相互作用は 0/1 数値の IT ダミーを使用（共線性軽減）
・Wald検定は scalar=True を明示
・summary2() を既定で使用（大規模固定効果の制約共分散警告を回避）

出力先:
  - 図: ./figures
  - 表: ./outputs
  - キャッシュCSV: ./cache
"""

import os, io, re, json, math, time, hashlib, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 日本語フォント設定
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc")
    fm.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
except Exception:
    pass

import matplotlib as mpl
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Noto Sans CJK JP"]
mpl.rcParams["axes.unicode_minus"] = False

# ============ ユーザー設定 ============

APP_ID = os.getenv("ESTAT_APP_ID", "e2b4cac5c935f29688d655174782322b024a6ad1")
STATS_DATA_ID = "0003445758"  # 令和2年以降 DB
LANG = "J"

# 対象年
TARGET_YEARS = [2020, 2021, 2022, 2023]

# IT職の簡易判定（必要に応じて調整）
IT_JOB_KEYWORDS = [
    "システム","SE","プログラ","ソフトウェア","情報処理","通信技術","ネットワーク",
    "インフラ","データ","AI","機械学習","QA（ソフト）","テスト（ソフト）",
    "システムコンサルタント","設計者","開発","運用","保守","セキュリティ","DB","SRE",
    "情報セキュリティ","クラウド","アーキテクト","PM（IT）","PM（情報）","IT"
]
EXCLUDE_SONOTA_FROM_IT = False  # 「その他」をITから外したければ True

# WLSの重み候補（見つかれば使用）
WEIGHT_METRIC_CANDIDATES = ["労働者数","労働者 数","就業者数","標本数","サンプル数"]

# 回帰の分散推定：都道府県クラスタ・ロバスト（失敗時はHC1）
PREF_CLUSTER_ROBUST = True

# 男女別の推定も出すか
RUN_BY_SEX = True

# 都道府県比較から「全国」を除くか
EXCLUDE_NATIONAL = True

# 保存先
SAVE_FIG = True
FIG_DIR = "./figures"
OUT_DIR = "./outputs"
CACHE_DIR = "./cache"

# KMeans クラスタリングの設定
K_CANDIDATES = [3, 4, 5, 6]  # シルエットで自動選択
RANDOM_STATE = 42

# ============ API エンドポイント ============

GET_STATS_DATA_JSON = "https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData"
GET_SIMPLE_CSV = "https://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData"

# ============ ユーティリティ ============

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def setup_japanese_font():
    try:
        from matplotlib import font_manager, rcParams
        candidates = [
            "Noto Sans CJK JP","Noto Sans JP",
            "IPAexGothic","IPAPGothic",
            "Hiragino Sans","Yu Gothic","MS Gothic","TakaoPGothic"
        ]
        avail = {f.name for f in font_manager.fontManager.ttflist}
        for name in candidates:
            if name in avail:
                rcParams["font.family"] = name
                rcParams["axes.unicode_minus"] = False
                print(f"[INFO] Matplotlib font set to: {name}")
                return
        print("[WARN] 日本語フォントが見当たりません（図が□になる可能性）。")
    except Exception as e:
        print("[WARN] フォント設定に失敗:", e)

def to_numeric_safe(x):
    if pd.isna(x): return np.nan
    s = re.sub(r"[^\d\.\-]", "", str(x))
    if s == "": return np.nan
    try: return float(s)
    except: return np.nan

def find_col(cols, keywords):
    for c in cols:
        for k in keywords:
            if k in c:
                return c
    return None

def cache_key(obj) -> str:
    j = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    import hashlib
    return hashlib.md5(j.encode("utf-8")).hexdigest()

# ============ メタから時間軸コード（cdTime）を取得 ============

def get_time_codes_map(stats_data_id):
    params = {
        "appId": APP_ID,
        "lang": LANG,
        "statsDataId": stats_data_id,
        "metaGetFlg": "Y",
        "cntGetFlg": "N",
        "sectionHeaderFlg": "1"
    }
    r = requests.get(GET_STATS_DATA_JSON, params=params, timeout=180)
    r.raise_for_status()
    js = r.json()
    try:
        class_obj = js["GET_STATS_DATA"]["STATISTICAL_DATA"]["CLASS_INF"]["CLASS_OBJ"]
        if isinstance(class_obj, dict):
            class_obj = [class_obj]
    except Exception:
        class_obj = []
    time_classes = []
    for obj in class_obj:
        _id = str(obj.get("@id","")).lower()
        _nm = str(obj.get("@name",""))
        if ("time" in _id) or ("時間" in _nm) or ("時間軸" in _nm) or (_id == "time"):
            classes = obj.get("CLASS", [])
            if isinstance(classes, dict):
                classes = [classes]
            time_classes.extend(classes)
    year_to_code = {}
    for c in time_classes:
        code = str(c.get("@code",""))
        name = str(c.get("@name",""))
        m = re.search(r"(19|20)\d{2}", name)
        if m:
            y = int(m.group(0))
            year_to_code[y] = code
    return year_to_code

# ============ CSV 取得（範囲→足りなければ年別） ============

def parse_csv_status(text: str):
    status = None; err = None
    for line in text.splitlines()[:300]:
        L = line.strip().strip('", ')
        if L.upper().startswith("STATUS"):
            m = re.search(r"STATUS[\",]*\s*[,]*\s*\"?(\d+)\"?", line, flags=re.I)
            if m: status = int(m.group(1))
        elif L.upper().startswith("ERROR_MSG"):
            m = re.search(r"ERROR_MSG[\",]*\s*,\s*\"(.*?)\"", line, flags=re.I)
            if m: err = m.group(1)
    return status, err

def robust_read_csv_from_text(text):
    status, err = parse_csv_status(text)
    if status is not None and status != 0:
        raise RuntimeError(f"STATUS={status}（{err or '原因不明'}）でVALUE表が出力されていません。")
    lines = text.splitlines()
    header_idx = None
    for i, line in enumerate(lines[:1200]):
        low = line.lower()
        if ("value" in low) and (("time_code" in low) or (" time" in low)) and (("area_code" in low) or ("area" in low)):
            header_idx = i; break
        if ("値" in line) and (("時間" in line) or ("時間軸" in line)) and (("地域" in line) or ("都道府県" in line)):
            header_idx = i; break
    if header_idx is None:
        head = "\n".join(lines[:30])
        raise RuntimeError("データヘッダ行を検出できませんでした。\n[先頭抜粋]\n"+head)
    csv_body = "\n".join(lines[header_idx:])
    return pd.read_csv(io.StringIO(csv_body), dtype=str, engine="python")

def http_get_csv(params):
    r = requests.get(GET_SIMPLE_CSV, params=params, timeout=180)
    r.raise_for_status()
    text = r.text
    ctype = (r.headers.get("Content-Type","") or "").lower()
    if "json" in ctype or text.strip().startswith("{"):
        try:
            info = r.json()
            st = info["GET_STATS_DATA"]["RESULT"]["STATUS"]
            em = info["GET_STATS_DATA"]["RESULT"]["ERROR_MSG"]
            raise RuntimeError(f"CSVではなくJSONが返りました（STATUS={st} / {em}）。")
        except Exception:
            raise RuntimeError(f"CSVではなくJSONが返りました（Content-Type={ctype}）。")
    return text

def fetch_df_range_or_yearly(stats_data_id, target_years, time_code_map):
    ensure_dir(CACHE_DIR)
    codes = [time_code_map[y] for y in target_years if y in time_code_map]
    uniq_codes = sorted(set(codes))
    if len(uniq_codes) == 0:
        print("[WARN] 対象年に対応する時間コードがメタから見つかりませんでした。年別取得に切り替えます。")
        return fetch_df_yearly_only(stats_data_id, target_years, time_code_map)

    code_min = min(int(c) for c in uniq_codes)
    code_max = max(int(c) for c in uniq_codes)

    params_range = {
        "appId": APP_ID, "lang": LANG, "statsDataId": stats_data_id,
        "metaGetFlg": "N", "cntGetFlg": "N",
        "explanationGetFlg": "N", "annotationGetFlg": "N",
        "sectionHeaderFlg": "2", "replaceSpChars": "1",
        "cdTimeFrom": str(code_min), "cdTimeTo": str(code_max)
    }
    ck = os.path.join(CACHE_DIR, f"simple_range_{stats_data_id}_{code_min}_{code_max}.csv")
    try:
        if os.path.exists(ck):
            df = pd.read_csv(ck, dtype=str)
        else:
            text = http_get_csv(params_range)
            df = robust_read_csv_from_text(text)
            df.to_csv(ck, index=False, encoding="utf-8")
        time_col_guess = find_col(df.columns, ["時間軸","年","time","時間軸（"])
        if time_col_guess:
            got_years = sorted(pd.to_numeric(
                pd.Series(df[time_col_guess].astype(str).str.extract(r"(\d{4})", expand=False)),
                errors="coerce"
            ).dropna().astype(int).unique().tolist())
            print(f"[INFO] 範囲取得で含まれた年: {got_years}")
        else:
            got_years = []
        missing = [y for y in target_years if (not got_years) or (y not in got_years)]
        if missing:
            print(f"[INFO] 範囲で欠けた年の年別取得を追加: {missing}")
            df_extra = fetch_df_yearly_only(stats_data_id, missing, time_code_map)
            if df_extra is not None and len(df_extra) > 0:
                df = pd.concat([df, df_extra], ignore_index=True, sort=False)
        return df
    except Exception as e:
        print(f"[INFO] 範囲取得に失敗。年別取得に切り替えます: {e}")
        return fetch_df_yearly_only(stats_data_id, target_years, time_code_map)

def fetch_df_yearly_only(stats_data_id, years, time_code_map):
    all_df = []
    for y in years:
        code = time_code_map.get(y)
        if not code:
            print(f"[WARN] year={y} の時間コードが見つからずスキップ")
            continue
        params_y = {
            "appId": APP_ID, "lang": LANG, "statsDataId": stats_data_id,
            "metaGetFlg": "N", "cntGetFlg": "N",
            "explanationGetFlg": "N", "annotationGetFlg": "N",
            "sectionHeaderFlg": "2", "replaceSpChars": "1",
            "cdTime": str(code)
        }
        ck = os.path.join(CACHE_DIR, f"simple_y{y}_{stats_data_id}_{code}.csv")
        try:
            if os.path.exists(ck):
                df_y = pd.read_csv(ck, dtype=str)
            else:
                print(f"[DL] year={y}, cdTime={code}")
                text = http_get_csv(params_y)
                df_y = robust_read_csv_from_text(text)
                df_y.to_csv(ck, index=False, encoding="utf-8")
            if not any(("年" in c) or ("時間" in c) or ("time" in c) for c in df_y.columns):
                df_y["年_推定"] = str(y)
            all_df.append(df_y)
        except Exception as e:
            print(f"[WARN] year={y} の取得に失敗: {e}")
        time.sleep(0.2)
    if not all_df:
        raise SystemExit("年別取得にも失敗しました。")
    return pd.concat(all_df, ignore_index=True, sort=False)

# ============ 整形・IT判定・特徴量 ============

def is_it_job(name: str) -> bool:
    if not isinstance(name, str): return False
    n = name.replace("　"," ").strip()
    hit = any(k in n for k in IT_JOB_KEYWORDS)
    if not hit: return False
    if EXCLUDE_SONOTA_FROM_IT and ("その他" in n): return False
    return True

def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    std = s.std(ddof=0)
    return (s - s.mean())/std if (not pd.isna(std) and std != 0) else s*0

def build_pivot(df_raw, sex_filter="男女計", exclude_national=True):
    df = df_raw.copy()
    df.columns = [c.replace("　"," ").strip() for c in df.columns]

    col_metric = find_col(df.columns, ["表章項目","Tabulation","Category"])
    col_value  = find_col(df.columns, ["値","VALUE","value"])
    col_unit   = find_col(df.columns, ["単位","UNIT","unit"])
    col_job    = find_col(df.columns, ["職種","職業","職種（小分類）","小分類"])
    col_pref   = find_col(df.columns, ["都道府県","地域","都道府県名","地域"])
    col_sex    = find_col(df.columns, ["性別","男女別","性別_基本"])
    col_time   = find_col(df.columns, ["時間軸","年","time","時間軸（"])

    if col_value:
        df[col_value] = df[col_value].map(to_numeric_safe)
    if sex_filter is not None and col_sex:
        df = df[df[col_sex] == sex_filter].copy()

    keys = [k for k in [col_job, col_pref, col_time] if k is not None]
    if not (col_metric and col_value and len(keys) > 0):
        raise RuntimeError("必要な列が不足（表章/値/職種or県or年）")

    pv = df.pivot_table(index=keys, columns=col_metric, values=col_value, aggfunc="first").reset_index()

    if col_time:
        pv["__year"] = pd.to_numeric(pv[col_time].astype(str).str.extract(r"(\d{4})", expand=False), errors="coerce")

    wage_metric_candidates = ["きまって支給する現金給与額","所定内給与","所定内給与額","所定内現金給与額","賃金"]
    cand_wage = [c for c in pv.columns for w in wage_metric_candidates if w in str(c)]
    wage_col = cand_wage[0] if cand_wage else None
    if wage_col is None:
        raise RuntimeError("賃金列が見つかりません（例：きまって支給する現金給与額 等）")

    add_age_cols   = [c for c in pv.columns if "年齢" in str(c)]
    add_ten_cols   = [c for c in pv.columns if "勤続" in str(c)]
    add_hour_cols  = [c for c in pv.columns if ("所定内実労働時間" in str(c)) or ("実労働時間" in str(c)) or (str(c)=="時間")]
    age_col   = add_age_cols[0] if add_age_cols else None
    ten_col   = add_ten_cols[0] if add_ten_cols else None
    hours_col = add_hour_cols[0] if add_hour_cols else None

    weight_cols = [c for c in pv.columns for w in WEIGHT_METRIC_CANDIDATES if w in str(c)]
    weight_col = weight_cols[0] if weight_cols else None

    if col_job:
        pv["IT職"] = pv[col_job].map(is_it_job)
    else:
        pv["IT職"] = False

    if exclude_national and col_pref and "全国" in pv[col_pref].astype(str).unique().tolist():
        pv = pv[pv[col_pref] != "全国"].copy()

    meta = {
        "col_metric": col_metric, "col_value": col_value, "col_unit": col_unit,
        "col_job": col_job, "col_pref": col_pref, "col_time": col_time,
        "wage_col": wage_col, "age_col": age_col, "ten_col": ten_col, "hours_col": hours_col,
        "weight_col": weight_col
    }
    return pv, meta

# ============ 回帰・EDA ============

def cov_fit_wls(formula, data, weights=None, cluster_groups=None):
    import statsmodels.formula.api as smf
    if weights is None:
        weights = np.ones(len(data))
    if cluster_groups is not None:
        try:
            return smf.wls(formula, data=data, weights=weights).fit(
                cov_type="cluster", cov_kwds={"groups": cluster_groups}, use_t=False
            )
        except Exception as e:
            print("[INFO] cluster失敗。HC1にフォールバック:", e)
    return smf.wls(formula, data=data, weights=weights).fit(cov_type="HC1", use_t=False)

def run_main_models(pv, meta, label_suffix="(All)"):
    col_pref = meta["col_pref"]; col_time = meta["col_time"]
    wage_col = meta["wage_col"]; weight_col = meta["weight_col"]

    df = pv.dropna(subset=[wage_col]).copy()
    df = df[np.isfinite(df[wage_col]) & (df[wage_col] > 0)]
    df["log_wage"] = np.log(df[wage_col])
    df["IT"] = df["IT職"].astype(int)  # ← 重要：相互作用の安定化

    if meta["age_col"]:   df["age_z"]   = zscore(df[meta["age_col"]])
    if meta["ten_col"]:   df["ten_z"]   = zscore(df[meta["ten_col"]])
    if meta["hours_col"]: df["hrs_z"]   = zscore(df[meta["hours_col"]])

    safe_pref = None; safe_time = None
    if col_pref:
        safe_pref = "__pref"; df[safe_pref] = df[col_pref].astype(str)
    if col_time:
        safe_time = "__time"
        yy = df[col_time].astype(str).str.extract(r"(\d{4})", expand=False)
        df[safe_time] = yy.where(yy.notna(), df[col_time].astype(str))

    weights = None
    if weight_col and (weight_col in df.columns):
        w = pd.to_numeric(df[weight_col], errors="coerce")
        if (w > 0).any():
            weights = w
        else:
            print("[INFO] weights候補は見つかったが正の値が無く、OLSとして実行")
    else:
        print("[INFO] weights列が見つからず、OLS（ロバスト分散）として実行")

    fe_pref = f" + C({safe_pref})" if safe_pref else ""
    fe_time = f" + C({safe_time})" if safe_time else ""
    ctrls = [c for c in ["age_z","ten_z","hrs_z"] if c in df.columns]
    ctrl_terms = (" + " + " + ".join(ctrls)) if ctrls else ""
    formula = "log_wage ~ IT" + ctrl_terms + fe_pref + fe_time
    print(f"\n=== Main WLS {label_suffix}: {formula} ===")

    cluster_groups = df[safe_pref] if (safe_pref and PREF_CLUSTER_ROBUST) else None
    res = cov_fit_wls(formula, df, weights=weights, cluster_groups=cluster_groups)

    # 概要
    try:
        print(res.summary2().as_text())
    except Exception:
        print(res.summary().as_text())

    # IT プレミアム
    name = "IT"
    if name in res.params.index:
        b = float(res.params[name]); se = float(res.bse[name])
        prem   = 100*(math.exp(b)-1)
        premL  = 100*(math.exp(b-1.96*se)-1)
        premH  = 100*(math.exp(b+1.96*se)-1)
        print(f"[{label_suffix} Main] IT賃金プレミアム: {prem:.2f}% (95%CI {premL:.2f}–{premH:.2f})")

    # 保存
    try:
        ensure_dir(OUT_DIR)
        res.params.to_csv(os.path.join(OUT_DIR, f"Main_params{label_suffix}.csv"), header=["coef"])
        res.bse.to_csv(os.path.join(OUT_DIR, f"Main_bse{label_suffix}.csv"), header=["se"])
        with open(os.path.join(OUT_DIR, f"Main_summary{label_suffix}.txt"), "w", encoding="utf-8") as f:
            try: f.write(res.summary2().as_text())
            except Exception: f.write(res.summary().as_text())
    except Exception as e:
        print("[WARN] 結果保存に失敗:", e)

    # 係数図
    try:
        coef = res.params.to_frame("coef").join(res.bse.to_frame("se"))
        coef["lo"] = coef["coef"] - 1.96*coef["se"]; coef["hi"] = coef["coef"] + 1.96*coef["se"]
        mask = ~coef.index.str.startswith("Intercept")
        top = coef[mask].reindex(coef[mask]["coef"].abs().sort_values(ascending=False).index).head(30)
        plt.figure()
        y = range(len(top))
        plt.hlines(y, top["lo"], top["hi"]); plt.plot(top["coef"], y, "o"); plt.axvline(0, ls="--")
        plt.yticks(y, top.index); plt.title(f"主要係数（Main, 95%CI）{label_suffix}"); plt.tight_layout()
        if SAVE_FIG:
            ensure_dir(FIG_DIR)
            out = os.path.join(FIG_DIR, f"coef_main_top30{label_suffix.replace(' ','_')}.png")
            plt.savefig(out); print(f"[SAVED] {out}")
        plt.close()
    except Exception as e:
        print("[WARN] 係数図の作成に失敗:", e)

    return res

# ============ 年別ITプレミアム推移 ============

def per_year_it_premium(pv, meta, label_suffix="(All)"):
    col_pref = meta["col_pref"]; col_time = meta["col_time"]
    wage_col = meta["wage_col"]; weight_col = meta["weight_col"]
    if col_time is None:
        print("[INFO] 年列が無いので per_year_it_premium をスキップ"); return None

    df = pv.dropna(subset=[wage_col]).copy()
    df = df[np.isfinite(df[wage_col]) & (df[wage_col] > 0)]
    df["log_wage"] = np.log(df[wage_col])
    df["IT"] = df["IT職"].astype(int)

    if meta["age_col"]:   df["age_z"]   = zscore(df[meta["age_col"]])
    if meta["ten_col"]:   df["ten_z"]   = zscore(df[meta["ten_col"]])
    if meta["hours_col"]: df["hrs_z"]   = zscore(df[meta["hours_col"]])

    ctrls = [c for c in ["age_z","ten_z","hrs_z"] if c in df.columns]
    ctrl_terms = (" + " + " + ".join(ctrls)) if ctrls else ""

    safe_pref = None
    if col_pref:
        safe_pref = "__pref"; df[safe_pref] = df[col_pref].astype(str)

    rows = []
    for y, sub in df.groupby(col_time):
        if len(sub) < 10: continue
        fe_pref = f" + C({safe_pref})" if safe_pref else ""
        formula = "log_wage ~ IT" + ctrl_terms + fe_pref

        weights = None
        if weight_col and (weight_col in sub.columns):
            _w = pd.to_numeric(sub[weight_col], errors="coerce")
            if (_w > 0).any(): weights = _w

        cluster_groups = sub[safe_pref] if (safe_pref and PREF_CLUSTER_ROBUST) else None
        res = cov_fit_wls(formula, sub, weights=weights, cluster_groups=cluster_groups)

        name = "IT"
        if name in res.params.index:
            b  = float(res.params[name]); se = float(res.bse[name])
            rows.append({
                "year": str(y),
                "beta": b, "se": se,
                "prem%": 100*(math.exp(b)-1),
                "premL%": 100*(math.exp(b-1.96*se)-1),
                "premH%": 100*(math.exp(b+1.96*se)-1),
            })

    if not rows:
        print("[INFO] 年別プレミアム推定に失敗またはデータ不足"); return None

    out_df = pd.DataFrame(rows).sort_values("year")
    ensure_dir(OUT_DIR)
    out_path = os.path.join(OUT_DIR, f"it_premium_by_year{label_suffix}.csv")
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig"); print(f"[SAVED] {out_path}")

    try:
        yy = pd.to_numeric(out_df["year"], errors="coerce")
        plt.figure()
        plt.plot(yy, out_df["prem%"], marker="o", label="IT賃金プレミアム（%）")
        plt.fill_between(yy, out_df["premL%"], out_df["premH%"], alpha=0.3)
        plt.axhline(0, ls="--")
        plt.title(f"IT賃金プレミアムの推移（{label_suffix}）")
        plt.xlabel("年"); plt.ylabel("ITプレミアム（%）")
        plt.tight_layout()
        if SAVE_FIG:
            ensure_dir(FIG_DIR)
            out = os.path.join(FIG_DIR, f"it_premium_trend{label_suffix.replace(' ','_')}.png")
            plt.savefig(out); print(f"[SAVED] {out}")
        plt.close()
    except Exception as e:
        print("[WARN] ITプレミアム推移図の作成に失敗:", e)

    return out_df

# ============ 都道府県トレンドランキング ============

def timeseries_eda(pv, meta, label_suffix="(All)"):
    col_time = meta["col_time"]; wage_col = meta["wage_col"]

    if SAVE_FIG:
        for flag, sub in pv.groupby("IT職"):
            plt.figure()
            sub[wage_col].dropna().plot(kind="hist", bins=40, alpha=0.9)
            plt.title(f"{wage_col} 分布（{'IT' if flag else '非IT'}・{label_suffix}）")
            plt.xlabel(wage_col); plt.ylabel("頻度")
            plt.tight_layout()
            ensure_dir(FIG_DIR)
            out = os.path.join(FIG_DIR, f"hist_{'IT' if flag else 'nonIT'}_{label_suffix}.png")
            plt.savefig(out); print(f"[SAVED] {out}")
            plt.close()

    if col_time:
        df = pv.dropna(subset=[wage_col]).copy()
        df["year"] = pd.to_numeric(df[col_time].astype(str).str.extract(r"(\d{4})", expand=False), errors="coerce")
        g = df.groupby(["year","IT職"], as_index=False)[wage_col].mean().rename(columns={wage_col:"mean_wage"})
        if len(g) > 0 and SAVE_FIG:
            plt.figure()
            for flag, sub in g.groupby("IT職"):
                label = "IT" if flag else "非IT"
                plt.plot(sub["year"], sub["mean_wage"], marker="o", label=label)
            plt.title(f"IT/非ITの賃金推移（平均, {label_suffix}）")
            plt.xlabel("年"); plt.ylabel(wage_col)
            plt.legend()
            plt.tight_layout()
            ensure_dir(FIG_DIR)
            out = os.path.join(FIG_DIR, f"mean_trend_it_nonit_{label_suffix}.png")
            plt.savefig(out); print(f"[SAVED] {out}")
            plt.close()

    if col_time and meta["col_pref"]:
        latest_year = pv[col_time].dropna().astype(str).max()
        it_latest = pv[(pv["IT職"]) & (pv[col_time] == latest_year)]
        if len(it_latest) > 0:
            pref_mean = it_latest.groupby(meta["col_pref"])[wage_col].mean().sort_values(ascending=False)
            print(f"\n=== IT職・都道府県平均（最新年: {latest_year}）Top10 ===")
            print(pref_mean.head(10))
            ensure_dir(OUT_DIR)
            pref_mean.to_csv(os.path.join(OUT_DIR, f"pref_mean_it_{latest_year}{label_suffix}.csv"),
                             header=["平均（単位は多くが千円/月）"], encoding="utf-8-sig")

def prefecture_trend_ranking(pv, meta, label_suffix="(All)", min_years=3):
    col_pref = meta["col_pref"]; col_time = meta["col_time"]; wage_col = meta["wage_col"]
    if not (col_pref and col_time):
        print("[INFO] 年 or 県が無いので都道府県トレンドをスキップ")
        return None

    df = pv[(pv["IT職"])].dropna(subset=[wage_col]).copy()
    df["year"] = pd.to_numeric(df[col_time].astype(str).str.extract(r"(\d{4})", expand=False), errors="coerce")
    df = df[np.isfinite(df[wage_col]) & (df[wage_col] > 0)]
    df["log_wage"] = np.log(df[wage_col])

    rows = []
    for pref, sub in df.groupby(col_pref):
        sub = sub.dropna(subset=["year"])
        if sub["year"].nunique() < min_years:
            continue
        x = sub["year"].values
        y = sub["log_wage"].values
        if len(x) < 2: continue
        A = np.vstack([x, np.ones(len(x))]).T
        try:
            slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
            rows.append({"pref": pref, "slope_pct_per_year": 100*(math.exp(slope)-1)})
        except Exception:
            pass

    if not rows:
        print("[INFO] IT×都道府県のトレンド推定に十分なデータがありません")
        return None

    out = pd.DataFrame(rows).sort_values("slope_pct_per_year", ascending=False)
    ensure_dir(OUT_DIR)
    path = os.path.join(OUT_DIR, f"pref_it_trend_ranking{label_suffix}.csv")
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {path}")

    try:
        top = out.head(10); bottom = out.tail(10)
        plt.figure(); plt.barh(top["pref"][::-1], top["slope_pct_per_year"][::-1])
        plt.title(f"IT賃金トレンド上位10（%/年, {label_suffix}）"); plt.tight_layout()
        if SAVE_FIG:
            ensure_dir(FIG_DIR)
            outp = os.path.join(FIG_DIR, f"pref_it_trend_top10{label_suffix}.png")
            plt.savefig(outp); print(f"[SAVED] {outp}")
        plt.close()

        plt.figure(); plt.barh(bottom["pref"][::-1], bottom["slope_pct_per_year"][::-1])
        plt.title(f"IT賃金トレンド下位10（%/年, {label_suffix}）"); plt.tight_layout()
        if SAVE_FIG:
            ensure_dir(FIG_DIR)
            outp = os.path.join(FIG_DIR, f"pref_it_trend_bottom10{label_suffix}.png")
            plt.savefig(outp); print(f"[SAVED] {outp}")
        plt.close()
    except Exception as e:
        print("[WARN] 都道府県トレンド図の作成に失敗:", e)

    return out

# ============ 都道府県クラスタリング & 相互作用回帰 ============

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def build_pref_features(pv, meta):
    """都道府県ごとの特徴量テーブルを作成（IT/非ITの賃金水準・差、年齢/勤続/時間の平均など）"""
    col_pref = meta["col_pref"]; col_time = meta["col_time"]; wage_col = meta["wage_col"]
    feats = []

    df = pv.copy()
    df = df.dropna(subset=[wage_col])
    if col_time:
        # 複数年を平均（直近の傾向も試したければ重み付け平均に変更可）
        pass

    for pref, sub in df.groupby(col_pref):
        if len(sub) == 0: continue
        it_wage   = sub.loc[sub["IT職"], wage_col].mean()
        non_wage  = sub.loc[~sub["IT職"], wage_col].mean()
        diff      = (it_wage - non_wage) if (pd.notna(it_wage) and pd.notna(non_wage)) else np.nan
        age = sub[meta["age_col"]].mean() if meta["age_col"] else np.nan
        ten = sub[meta["ten_col"]].mean() if meta["ten_col"] else np.nan
        hrs = sub[meta["hours_col"]].mean() if meta["hours_col"] else np.nan
        overall   = sub[wage_col].mean()
        it_share  = sub["IT職"].mean()  # 行単位のIT比率（簡易 proxy）

        feats.append({
            "pref": pref,
            "wage_overall": overall,
            "wage_it": it_wage,
            "wage_nonit": non_wage,
            "wage_it_minus_non": diff,
            "age_mean": age,
            "ten_mean": ten,
            "hrs_mean": hrs,
            "it_share": it_share
        })
    feat_df = pd.DataFrame(feats).set_index("pref")
    return feat_df

def choose_k_and_fit_kmeans(feat_df, k_candidates=K_CANDIDATES, random_state=RANDOM_STATE):
    X = feat_df.copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.values)

    best = None
    for k in k_candidates:
        km = KMeans(n_clusters=k, n_init=20, random_state=random_state)
        labels = km.fit_predict(Xs)
        sil = silhouette_score(Xs, labels)
        if (best is None) or (sil > best["sil"]):
            best = {"k": k, "sil": sil, "labels": labels, "km": km, "scaler": scaler}
    return best, X.columns.tolist()

import pathlib
def export_cluster_mapping(df, tag="(All)"):
    """
    df: 都道府県ダミー __pref と クラスタ番号 _cluster を含む分析用の長いデータ
    tag: ファイル名に付けるラベル 例) "(All)" "(男)" "(女)"
    出力:
      ./outputs/cluster_mapping{tag}.csv
      ./outputs/cluster_members{tag}.txt
    """
    outdir = pathlib.Path(OUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    mapping = (
        df.loc[:, ["__pref", "_cluster"]]
          .dropna()
          .drop_duplicates()
          .assign(_cluster=lambda x: x["_cluster"].astype(int))
          .sort_values(["_cluster", "__pref"])
          .rename(columns={"__pref": "地域", "_cluster": "cluster"})
    )

    csv_path = outdir / f"cluster_mapping{tag}.csv"
    mapping.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {csv_path}")

    txt_path = outdir / f"cluster_members{tag}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        for cid, g in mapping.groupby("cluster"):
            line = f"Cluster {cid}: " + "、".join(g["地域"].tolist())
            print(line)
            f.write(line + "\n")
    print(f"[SAVED] {txt_path}")

def run_with_cluster_interaction(pv, meta, pref2cluster, label_suffix="(All+CLxIT)"):
    """IT(数値) × 都道府県クラスタ の相互作用回帰と、クラスタ別ITプレミアムの推定"""
    import statsmodels.formula.api as smf

    col_pref = meta["col_pref"]; col_time = meta["col_time"]
    wage_col = meta["wage_col"]; weight_col = meta["weight_col"]

    df = pv.dropna(subset=[wage_col]).copy()
    df = df[np.isfinite(df[wage_col]) & (df[wage_col] > 0)]
    df["log_wage"] = np.log(df[wage_col])
    df["IT"] = df["IT職"].astype(int)  # 相互作用安定化のため bool→0/1

    if meta["age_col"]:   df["age_z"]   = zscore(df[meta["age_col"]])
    if meta["ten_col"]:   df["ten_z"]   = zscore(df[meta["ten_col"]])
    if meta["hours_col"]: df["hrs_z"]   = zscore(df[meta["hours_col"]])

    safe_pref = "__pref"; safe_time = "__time"
    df[safe_pref] = df[col_pref].astype(str)
    yy = df[col_time].astype(str).str.extract(r"(\d{4})", expand=False)
    df[safe_time] = yy.where(yy.notna(), df[col_time].astype(str))

    # クラスタラベルを付与
    df["_cluster"] = df[col_pref].map(pref2cluster).astype("category")

    # ここでクラスタ対応表を出力
    export_cluster_mapping(df, tag=label_suffix)

    cats = list(df["_cluster"].cat.categories)

    weights = None
    if weight_col and (weight_col in df.columns):
        w = pd.to_numeric(df[weight_col], errors="coerce")
        if (w > 0).any(): weights = w

    ctrls = [c for c in ["age_z","ten_z","hrs_z"] if c in df.columns]
    ctrl_terms = (" + " + " + ".join(ctrls)) if ctrls else ""

    formula = "log_wage ~ IT" + ctrl_terms + " + C(__pref) + C(__time) + IT:C(_cluster)"
    print(f"\n=== Interaction WLS {label_suffix}: {formula} ===")
    res = smf.wls(formula, data=df, weights=(weights if weights is not None else np.ones(len(df)))).fit(
        cov_type="cluster", cov_kwds={"groups": df[safe_pref]}, use_t=False
    )

    try:
        print(res.summary2().as_text())
    except Exception:
        print(res.summary().as_text())

    cov = res.cov_params()
    idx = list(res.params.index)
    name_base = "IT"
    out_rows = []
    for g in cats:
        if g == cats[0]:
            L = np.zeros(len(idx)); L[idx.index(name_base)] = 1.0
        else:
            iname = next((cand for cand in (f"C(_cluster)[T.{g}]:IT", f"IT:C(_cluster)[T.{g}]") if cand in idx), None)
            if iname is None: continue
            L = np.zeros(len(idx))
            L[idx.index(name_base)] = 1.0
            L[idx.index(iname)] = 1.0

        beta = float(L @ res.params.values)
        se   = float((L @ cov.values @ L.T) ** 0.5)
        prem = 100*(math.exp(beta)-1)
        lo   = 100*(math.exp(beta-1.96*se)-1)
        hi   = 100*(math.exp(beta+1.96*se)-1)
        out_rows.append({"cluster": int(g), "beta": beta, "se": se, "prem%": prem, "premL%": lo, "premH%": hi})

    out = pd.DataFrame(out_rows).sort_values("cluster")
    ensure_dir(OUT_DIR)
    out_path = os.path.join(OUT_DIR, f"it_premium_by_cluster{label_suffix}.csv")
    out.to_csv(out_path, index=False, encoding="utf-8-sig"); print(f"[SAVED] {out_path}")

    print("\n=== クラスター別 IT プレミアム（%） ===")
    print(out[["cluster","prem%","premL%","premH%"]])

    terms = [n for n in idx if ("C(_cluster)" in n) and (":IT" in n or "IT:" in n)]
    if terms:
        R = np.zeros((len(terms), len(res.params)))
        for r, name in enumerate(terms):
            R[r, idx.index(name)] = 1.0
        print("\n=== Wald test: 相互作用すべて=0（タイプ差なし） ===")
        print(res.wald_test(R, scalar=True))
    else:
        print("[INFO] 相互作用項がモデルに含まれていません。")

    return res, out

# ============ メイン ============

def main():
    if not APP_ID or APP_ID.strip() == "":
        raise SystemExit("環境変数 ESTAT_APP_ID を設定してください。")

    ensure_dir(FIG_DIR); ensure_dir(OUT_DIR); ensure_dir(CACHE_DIR)
    setup_japanese_font()

    print("[INFO] 時間コードを取得中 ...")
    year_code_map = get_time_codes_map(STATS_DATA_ID)
    print("[INFO] メタで見つかった年→code:", {k: year_code_map.get(k) for k in sorted(year_code_map.keys())})
    missing_years = [y for y in TARGET_YEARS if y not in year_code_map]
    if missing_years:
        print(f"[WARN] メタに存在しない年（スキップ対象）: {missing_years}")

    print(f"[INFO] 取得対象年: {TARGET_YEARS}")
    df_raw = fetch_df_range_or_yearly(STATS_DATA_ID, TARGET_YEARS, year_code_map)

    print("Columns:", list(df_raw.columns))
    col_time_guess = find_col(df_raw.columns, ["時間軸","年","time","時間軸（"])
    if col_time_guess:
        s_year = df_raw[col_time_guess].astype(str).str.extract(r"(\d{4})", expand=False)
        years_list = pd.to_numeric(s_year, errors="coerce").dropna().astype(int).unique().tolist()
        years_list.sort()
        with open(os.path.join(OUT_DIR, "years_unique.txt"), "w", encoding="utf-8") as f:
            f.write(", ".join(map(str, years_list)))
        print("Unique years in raw:", years_list)
        missing = sorted(set(TARGET_YEARS) - set(years_list))
        if missing:
            print(f"[WARN] 取り込めていない年があります: {missing}")
    else:
        print("[WARN] 時間列が見当たりません（年の保険列を活用している可能性）")

    # 男女計（メイン）
    pv, meta = build_pivot(df_raw, sex_filter="男女計", exclude_national=EXCLUDE_NATIONAL)
    if meta["col_job"]:
        top_jobs = pv[meta["col_job"]].value_counts().head(20)
        print("\n=== 職種Top20（出現頻度, 男女計） ===")
        print(top_jobs)
        top_jobs.to_csv(os.path.join(OUT_DIR, "top_jobs_all.csv"), encoding="utf-8-sig")
    timeseries_eda(pv, meta, label_suffix="(All)")
    run_main_models(pv, meta, label_suffix="(All)")
    per_year_it_premium(pv, meta, label_suffix="(All)")
    prefecture_trend_ranking(pv, meta, label_suffix="(All)", min_years=3)

    # クラスタリング → 相互作用
    print("\n[STEP] 都道府県の特徴量を作成してクラスタリングします …")
    feat_df = build_pref_features(pv, meta)
    best, feat_names = choose_k_and_fit_kmeans(feat_df, k_candidates=K_CANDIDATES, random_state=RANDOM_STATE)
    print(f"[INFO] クラスタリング最適候補: k={best['k']}, silhouette={best['sil']:.3f}")
    labels = best["labels"]
    pref2cluster = {pref: int(lbl) for pref, lbl in zip(feat_df.index.tolist(), labels)}
    run_with_cluster_interaction(pv, meta, pref2cluster, label_suffix="(All+CLxIT)")

    # 男女別
    if RUN_BY_SEX:
        for sex in ["男", "女"]:
            try:
                pv_s, meta_s = build_pivot(df_raw, sex_filter=sex, exclude_national=EXCLUDE_NATIONAL)
                print(f"\n--- 性別={sex} ---")
                timeseries_eda(pv_s, meta_s, label_suffix=f"({sex})")
                run_main_models(pv_s, meta_s, label_suffix=f"({sex})")
                per_year_it_premium(pv_s, meta_s, label_suffix=f"({sex})")
                run_with_cluster_interaction(pv_s, meta_s, pref2cluster, label_suffix=f"({sex}+CLxIT)")
            except Exception as e:
                print(f"[WARN] 性別={sex} の処理に失敗: {e}")

    print("\nDone. 図は ./figures、表は ./outputs に保存しました。")

if __name__ == "__main__":
    main()


[INFO] Matplotlib font set to: Noto Sans CJK JP
[INFO] 時間コードを取得中 ...
[INFO] メタで見つかった年→code: {2020: '2020000000', 2021: '2021000000', 2022: '2022000000', 2023: '2023000000'}
[INFO] 取得対象年: [2020, 2021, 2022, 2023]
[INFO] 範囲取得で含まれた年: [2022]
[INFO] 範囲で欠けた年の年別取得を追加: [2020, 2021, 2023]
[DL] year=2020, cdTime=2020000000
[DL] year=2021, cdTime=2021000000
[DL] year=2023, cdTime=2023000000
Columns: ['tab_code', '表章項目', 'cat01_code', '性別_基本', 'cat02_code', '職種（小分類）（2020～）', 'area_code', '地域', 'time_code', '時間軸（2020～2023）', 'unit', 'value']
Unique years in raw: [2020, 2021, 2022, 2023]

=== 職種Top20（出現頻度, 男女計） ===
職種（小分類）（2020～）
その他のサービス職業従事者                     188
その他の一般事務従事者                       188
その他の保健医療サービス職業従事者                 188
その他の保健医療従事者                       188
その他の保安職業従事者                       188
その他の商品販売従事者                       188
その他の営業職業従事者                       188
その他の定置・建設機械運転従事者                  188
その他の建設従事者                         188
その他の情報処理・通信技術者                    

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 53, but rank is 7
  warnings.warn('covariance of constraints does not have full '


                  Results: Weighted least squares
Model:                WLS              Adj. R-squared:     0.162    
Dependent Variable:   log_wage         AIC:                2265.2916
Date:                 2025-08-02 09:47 BIC:                2706.5541
No. Observations:     26148            Log-Likelihood:     -1078.6  
Df Model:             53               F-statistic:        2162.    
Df Residuals:         26094            Prob (F-statistic): 1.08e-55 
R-squared:            0.164            Scale:              0.063717 
--------------------------------------------------------------------
                   Coef.  Std.Err.     z      P>|z|   [0.025  0.975]
--------------------------------------------------------------------
Intercept          5.7449   0.0015  3860.2457 0.0000  5.7419  5.7478
C(__pref)[T.京都府]   0.0192   0.0003    70.0477 0.0000  0.0186  0.0197
C(__pref)[T.佐賀県]  -0.1295   0.0003  -406.6571 0.0000 -0.1301 -0.1288
C(__pref)[T.兵庫県]   0.0301   0.0004    83.3743 0.0000 

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 56, but rank is 10
  warnings.warn('covariance of constraints does not have full '


                   Results: Weighted least squares
Model:               WLS               Adj. R-squared:      0.162    
Dependent Variable:  log_wage          AIC:                 2269.8045
Date:                2025-08-02 09:47  BIC:                 2735.5815
No. Observations:    26148             Log-Likelihood:      -1077.9  
Df Model:            56                F-statistic:         1575.    
Df Residuals:        26091             Prob (F-statistic):  8.28e-55 
R-squared:           0.164             Scale:               0.063721 
---------------------------------------------------------------------
                     Coef.  Std.Err.     z     P>|z|   [0.025  0.975]
---------------------------------------------------------------------
Intercept            5.7447   0.0015 3730.7343 0.0000  5.7416  5.7477
C(__pref)[T.京都府]     0.0191   0.0003   69.7361 0.0000  0.0186  0.0197
C(__pref)[T.佐賀県]    -0.1291   0.0005 -240.5165 0.0000 -0.1302 -0.1281
C(__pref)[T.兵庫県]     0.0299   0.0005   

...

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 56, but rank is 10
  warnings.warn('covariance of constraints does not have full '


                   Results: Weighted least squares
Model:               WLS               Adj. R-squared:      0.151    
Dependent Variable:  log_wage          AIC:                 1274.6411
Date:                2025-08-02 09:47  BIC:                 1715.7958
No. Observations:    16976             Log-Likelihood:      -580.32  
Df Model:            56                F-statistic:         411.2    
Df Residuals:        16919             Prob (F-statistic):  1.72e-41 
R-squared:           0.154             Scale:               0.062904 
---------------------------------------------------------------------
                     Coef.  Std.Err.     z     P>|z|   [0.025  0.975]
---------------------------------------------------------------------
Intercept            5.8226   0.0015 3789.4864 0.0000  5.8196  5.8256
C(__pref)[T.京都府]     0.0110   0.0004   28.8688 0.0000  0.0102  0.0117
C(__pref)[T.佐賀県]    -0.1177   0.0008 -151.8426 0.0000 -0.1192 -0.1162
C(__pref)[T.兵庫県]     0.0210   0.0007   

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 53, but rank is 7
  warnings.warn('covariance of constraints does not have full '


                  Results: Weighted least squares
Model:               WLS              Adj. R-squared:     0.114    
Dependent Variable:  log_wage         AIC:                4359.8833
Date:                2025-08-02 09:47 BIC:                4778.0320
No. Observations:    17043            Log-Likelihood:     -2125.9  
Df Model:            53               F-statistic:        1095.    
Df Residuals:        16989            Prob (F-statistic): 6.22e-49 
R-squared:           0.117            Scale:              0.075379 
-------------------------------------------------------------------
                   Coef.  Std.Err.     z     P>|z|   [0.025  0.975]
-------------------------------------------------------------------
Intercept          5.5528   0.0018 3078.3182 0.0000  5.5492  5.5563
C(__pref)[T.京都府]   0.0487   0.0004  127.2164 0.0000  0.0480  0.0495
C(__pref)[T.佐賀県]  -0.1083   0.0010 -112.5866 0.0000 -0.1102 -0.1064
C(__pref)[T.兵庫県]   0.0346   0.0004   88.4153 0.0000  0.0339  0.035

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 56, but rank is 10
  warnings.warn('covariance of constraints does not have full '
